# Skin Condition Detection — Production Pipeline v3 (PyTorch)

## What changed from v2
| Area | v2 | v3 |
|------|----|----|
| Overfitting | High train/val gap | SWA + CutMix + simpler head + higher reg |
| Head | Dropout→512→BN→SiLU→Dropout→Linear | Dropout(0.4)→GeM→Linear (minimal, no memorisation) |
| Augmentation | RandAugment mag=9, no CutMix | mag=7, CutMix alternated with MixUp |
| Backbone choice | Only EfficientNet-B2 ImageNet | 3 options: Standard / Noisy-Student / DINOv2 |
| Skin-specific pretraining | ❌ | ✅ DINOv2 option (proven in derm papers) + Noisy-Student |
| Scheduler | Cosine warmup | Cosine warmup + **SWA** (smooths sharp minima) |
| Weight decay | 1e-4 | 5e-4 |
| Label smoothing | 0.05 | 0.10 |
| Gradient accumulation | ❌ | ✅ (effective batch = batch_size × accum_steps) |
| Model summary | ❌ | ✅ torchinfo layer-by-layer table |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# COLAB FLAG  →  set IS_COLAB = True when running on Google Colab
# ─────────────────────────────────────────────────────────────────────────
IS_COLAB = False   # <── flip to True on Colab

if IS_COLAB:
    import zipfile
    from pathlib import Path as _P

    _ZIP  = _P('/content/dataset.zip')
    _DEST = _P('/content/dataset')

    if not _ZIP.exists():
        raise FileNotFoundError(
            'dataset.zip not found at /content/dataset.zip.\n'
            'Upload it via the Colab Files panel or mount Google Drive first.'
        )

    if _DEST.exists():
        print(f'Already extracted → {_DEST}, skipping.')
    else:
        print(f'Extracting {_ZIP} ...')
        with zipfile.ZipFile(_ZIP, 'r') as zf:
            zf.extractall(_DEST)
        print('Done.')

    # Handle zip with a single top-level folder inside
    _subdirs = [p for p in _DEST.iterdir() if p.is_dir()]
    if len(_subdirs) == 1 and not (_DEST / 'Acne').exists():
        _DATASET_ROOT = str(_subdirs[0])
        print(f'Inner folder detected, using: {_DATASET_ROOT}')
    else:
        _DATASET_ROOT = str(_DEST)

    OUTPUT_DIR = '/content/artifacts'
else:
    _DATASET_ROOT = '/home/dev/Desktop/cnn/dataset'   # <── change to your local path
    OUTPUT_DIR    = '/home/dev/Desktop/cnn/artifacts'

print('Dataset root :', _DATASET_ROOT)
print('Output dir   :', OUTPUT_DIR)


## Backbone Choice

Pick **one** of the three options below by setting `BACKBONE_CHOICE`.

| Option | Model | Pretrained on | Notes |
|--------|-------|--------------|-------|
| `'standard'` | EfficientNet-B2 | ImageNet-1k (1.2M images) | Your current setup |
| `'noisy_student'` | EfficientNet-B2 NS | JFT-300M (300M images, pseudo-labels) | ~2-3% better, same arch |
| `'dinov2'` | ViT-S/14 DINOv2 | LVD-142M (self-supervised, 142M images) | **Best option for skin**: self-supervised features proven strong in dermatology papers. Requires `pip install timm` |

> **Why DINOv2 for skin?** DINOv2 is a self-supervised ViT pretrained on 142M images with no labels. Because it learns dense spatial features via self-distillation, it captures fine-grained texture patterns (critical for distinguishing acne types, keratosis, vitiligo patches, etc.) far better than ImageNet-supervised CNNs. Multiple dermatology papers (2024) report DINOv2 outperforming EfficientNet-pretrained models on skin datasets with identical fine-tuning budgets.

> **Why NOT Google Derm Foundation?** It is TensorFlow-only and produces frozen embeddings — you cannot fine-tune the backbone in PyTorch. **PanDerm** (2M skin images) is research-only and not publicly released.

> **Noisy Student** is the easiest upgrade if you want to stick with a CNN.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# BACKBONE_CHOICE:  'standard'  |  'noisy_student'  |  'dinov2'
# ─────────────────────────────────────────────────────────────────────────
BACKBONE_CHOICE = 'noisy_student'   # <── change here

assert BACKBONE_CHOICE in ('standard', 'noisy_student', 'dinov2'), \
    f'Unknown backbone: {BACKBONE_CHOICE}'

print(f'Backbone selected: {BACKBONE_CHOICE}')


In [ ]:
# Uncomment once to install deps:
# !pip install -q torch torchvision timm torchinfo scikit-learn seaborn pandas numpy matplotlib tqdm pillow

import os, json, time, copy, math, random
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import datasets, transforms, models

try:
    import timm
    HAS_TIMM = True
    print(f'timm {timm.__version__} available')
except ImportError:
    HAS_TIMM = False
    print('timm not found — run: pip install timm  (required for noisy_student / dinov2)')

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except ImportError:
    HAS_TORCHINFO = False
    print('torchinfo not found — run: pip install torchinfo  (required for model summary)')

try:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print('scikit-learn / seaborn not found — install for full eval report')

# Validate backbone deps
if BACKBONE_CHOICE in ('noisy_student', 'dinov2') and not HAS_TIMM:
    raise ImportError('timm is required for noisy_student and dinov2 backbones. Run: pip install timm')

print(f'\nPyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')


In [ ]:
@dataclass
class Config:
    # Paths
    dataset_root: str = _DATASET_ROOT
    output_dir:   str = OUTPUT_DIR
    run_name:     str = f'skin_{BACKBONE_CHOICE}'

    # Image / loader  (img_size auto-set below based on backbone)
    img_size:    int = 224   # overridden below
    batch_size:  int = 32
    num_workers: int = 2

    # Training
    epochs:        int   = 40
    lr:            float = 2e-4
    weight_decay:  float = 5e-4    # stronger reg vs v2 (was 1e-4)
    label_smoothing: float = 0.10  # stronger label smoothing (was 0.05)
    warmup_epochs: int   = 3
    freeze_backbone_epochs: int = 2

    # Gradient accumulation: effective_batch = batch_size * accum_steps
    accum_steps: int = 2

    # Augmentation
    mixup_alpha:  float = 0.4    # higher than v2
    cutmix_alpha: float = 1.0    # new in v3 — alternated with mixup
    cutmix_prob:  float = 0.5    # probability of choosing cutmix vs mixup

    # Imbalance handling
    use_weighted_sampler:    bool  = True
    use_focal_loss:          bool  = True
    focal_gamma:             float = 2.0
    use_class_weighted_loss: bool  = True

    # SWA (Stochastic Weight Averaging) — key anti-overfit technique
    use_swa:       bool  = True
    swa_start:     int   = 20    # epoch to begin averaging
    swa_lr:        float = 5e-5

    # Early stopping
    patience:  int   = 10
    min_delta: float = 1e-4

    seed: int = 42


cfg = Config()

# Auto-set img_size by backbone
if BACKBONE_CHOICE == 'dinov2':
    cfg.img_size = 224   # DINOv2 ViT-S/14 → multiples of 14; 224=16×14
elif BACKBONE_CHOICE == 'noisy_student':
    cfg.img_size = 260   # EfficientNet-B2 native resolution
else:
    cfg.img_size = 260

assert abs((0.75 + 0.15 + 0.10) - 1.0) < 1e-8

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device    :', DEVICE)
print('img_size  :', cfg.img_size)
print('Backbone  :', BACKBONE_CHOICE)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

RUN_DIR = Path(cfg.output_dir) / cfg.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run dir   :', RUN_DIR)


In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────
# v3 vs v2:  RandAugment magnitude reduced 9→7 to avoid excessive distortion
#            that causes weighted-sampler minority repeats to still look "easy"

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.65, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), shear=8),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandAugment(num_ops=2, magnitude=7),   # reduced from 9
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.12), ratio=(0.3, 3.3), value='random'),
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(cfg.img_size * 1.14)),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# 3-view TTA used at inference
tta_tfms = [
    eval_tfms,
    transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.14)),
        transforms.CenterCrop(cfg.img_size),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
    transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.25)),
        transforms.CenterCrop(cfg.img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]),
]

print('Augmentation pipeline ready.')


In [ ]:
base_dataset = datasets.ImageFolder(cfg.dataset_root)
class_names  = base_dataset.classes
num_classes  = len(class_names)
print('Classes    :', class_names)
print('Num classes:', num_classes)

with open(RUN_DIR / 'class_to_idx.json', 'w') as f:
    json.dump(base_dataset.class_to_idx, f, indent=2)

# ── Stratified split ────────────────────────────────────────────────────────
targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))
by_class = defaultdict(list)
for idx, y in zip(indices, targets):
    by_class[int(y)].append(int(idx))

train_idx, val_idx, test_idx = [], [], []
for cls, cls_indices in by_class.items():
    cls_indices = np.array(cls_indices)
    rng = np.random.default_rng(cfg.seed + cls)
    rng.shuffle(cls_indices)
    n = len(cls_indices)
    if n == 1:
        n_train, n_val, n_test = 1, 0, 0
    elif n == 2:
        n_train, n_val, n_test = 1, 0, 1
    else:
        n_train = max(1, int(round(n * 0.75)))
        n_val   = max(1, int(round(n * 0.15)))
        n_test  = n - n_train - n_val
        if n_test < 1:
            take = min(1 - n_test, max(0, n_train - 1))
            n_train -= take
            n_test = n - n_train - n_val
    train_idx.extend(cls_indices[:n_train].tolist())
    val_idx.extend(  cls_indices[n_train:n_train + n_val].tolist())
    test_idx.extend( cls_indices[n_train + n_val:n_train + n_val + n_test].tolist())

if not val_idx:  val_idx.append(train_idx.pop())
if not test_idx: test_idx.append(train_idx.pop())

print(f'Train/Val/Test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}')

train_dataset_full = datasets.ImageFolder(cfg.dataset_root, transform=train_tfms)
val_dataset_full   = datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms)
test_dataset_full  = datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms)

train_subset = Subset(train_dataset_full, train_idx)
val_subset   = Subset(val_dataset_full,   val_idx)
test_subset  = Subset(test_dataset_full,  test_idx)

train_targets = targets[train_idx]
val_targets   = targets[val_idx]
test_targets  = targets[test_idx]

dist_df = pd.DataFrame({
    'class': class_names,
    'train': np.bincount(train_targets, minlength=num_classes),
    'val':   np.bincount(val_targets,   minlength=num_classes),
    'test':  np.bincount(test_targets,  minlength=num_classes),
})
display(dist_df)


In [ ]:
# ── Imbalance handling ────────────────────────────────────────────────────
# v3 change: use SQRT-based weights instead of pure inverse frequency.
# This is less aggressive — avoids over-repeating the tiny classes (Whiteheads=72)
# to the point where the model memorises their augmented versions.

class_counts_train = np.bincount(train_targets, minlength=num_classes).astype(float)
max_count = class_counts_train.max()

# sqrt( max / class_count ) — gentler than 1/count
sample_weights_cls = np.where(
    class_counts_train > 0,
    np.sqrt(max_count / np.maximum(class_counts_train, 1)),
    0.0,
)
# Cap at 5× to prevent pathological over-sampling of 2-digit classes
sample_weights_cls = np.minimum(sample_weights_cls, 5.0)

# Class-weighted loss uses same sqrt weights (normalised)
class_weights_loss = sample_weights_cls / sample_weights_cls.mean()

print('Class counts (train):', class_counts_train.astype(int).tolist())
print('Sampler weights     :', sample_weights_cls.round(3).tolist())
print('Loss weights        :', class_weights_loss.round(3).tolist())

sample_weights = sample_weights_cls[train_targets]
train_sampler = None
train_shuffle = True
if cfg.use_weighted_sampler:
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_shuffle = False

_nw = cfg.num_workers
_pm = (DEVICE == 'cuda')
_pw = (_nw > 0)

train_loader = DataLoader(train_subset, batch_size=cfg.batch_size,
                          shuffle=train_shuffle, sampler=train_sampler,
                          num_workers=_nw, pin_memory=_pm, persistent_workers=_pw)
val_loader   = DataLoader(val_subset,   batch_size=cfg.batch_size,
                          shuffle=False, num_workers=_nw, pin_memory=_pm, persistent_workers=_pw)
test_loader  = DataLoader(test_subset,  batch_size=cfg.batch_size,
                          shuffle=False, num_workers=_nw, pin_memory=_pm, persistent_workers=_pw)

print(f'\nLoaders: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)} batches')


In [ ]:
# ── Focal Loss ────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits, targets, weight=self.weight,
            label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-F.cross_entropy(logits, targets, reduction='none'))
        return ((1 - pt) ** self.gamma * ce_loss).mean()


_cw = (torch.tensor(class_weights_loss, dtype=torch.float32, device=DEVICE)
       if cfg.use_class_weighted_loss else None)

criterion = (FocalLoss(gamma=cfg.focal_gamma, weight=_cw,
                       label_smoothing=cfg.label_smoothing)
             if cfg.use_focal_loss
             else nn.CrossEntropyLoss(weight=_cw, label_smoothing=cfg.label_smoothing))

print(f'Loss: {criterion.__class__.__name__}' +
      (f' (gamma={cfg.focal_gamma})' if cfg.use_focal_loss else ''))


In [ ]:
# ── GeM Pooling ───────────────────────────────────────────────────────────
# Generalised Mean Pooling outperforms avg/max for fine-grained recognition.
# Used only with EfficientNet variants; DINOv2 already uses CLS token.
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        # x: (B, C, H, W)
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),
                            (x.size(-2), x.size(-1))).pow(1.0 / self.p)


# ── Build backbone ────────────────────────────────────────────────────────
def build_model(backbone_choice, num_classes, device):
    if backbone_choice == 'standard':
        # Original: EfficientNet-B2 from torchvision, ImageNet-1k weights
        base = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
        in_features = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(p=0.40, inplace=True),
            nn.Linear(in_features, num_classes),
        )
        return base.to(device), base.features

    elif backbone_choice == 'noisy_student':
        # EfficientNet-B2 trained with Noisy Student on JFT-300M (300M images).
        # Same architecture as standard but with dramatically richer features —
        # essentially free upgrade, no code change needed at inference.
        base = timm.create_model(
            'tf_efficientnet_b2.ns_jft_in1k',
            pretrained=True,
            num_classes=0,   # remove head, we'll add our own
        )
        in_features = base.num_features
        head = nn.Sequential(
            GeM(),
            nn.Flatten(),
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=0.40),
            nn.Linear(in_features, num_classes),
        )
        model = nn.Sequential(base, head)

        class _NSWrapper(nn.Module):
            def __init__(self, backbone, head):
                super().__init__()
                self.backbone = backbone
                self.head = head
            def forward(self, x):
                feat = self.backbone.forward_features(x)
                return self.head(feat)

        model = _NSWrapper(base, head)
        model = model.to(device)
        return model, base

    elif backbone_choice == 'dinov2':
        # DINOv2 ViT-S/14 pretrained on LVD-142M via self-supervised learning.
        # Self-supervised features capture fine-grained textures without requiring
        # labelled skin data — proven very strong in 2024 dermatology papers.
        # ViT-S: 22M params (fast to train), ViT-B: 86M (stronger).
        base = timm.create_model(
            'vit_small_patch14_reg4_dinov2.lvd142m',
            pretrained=True,
            num_classes=0,   # remove classification head
            img_size=cfg.img_size,
        )
        in_features = base.num_features
        head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(p=0.40),
            nn.Linear(in_features, num_classes),
        )

        class _DINOWrapper(nn.Module):
            def __init__(self, backbone, head):
                super().__init__()
                self.backbone = backbone
                self.head = head
            def forward(self, x):
                feat = self.backbone(x)   # CLS token output
                return self.head(feat)

        model = _DINOWrapper(base, head).to(device)
        return model, base

    else:
        raise ValueError(f'Unknown backbone: {backbone_choice}')


model, _backbone_module = build_model(BACKBONE_CHOICE, num_classes, DEVICE)

# Freeze backbone for warmup epochs
for p in _backbone_module.parameters():
    p.requires_grad = False

print(f'Backbone frozen for first {cfg.freeze_backbone_epochs} epochs.')
print(model)


In [ ]:
# ── Model Summary ─────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print('=' * 65)
print(f'  BACKBONE  : {BACKBONE_CHOICE}')
print(f'  IMG SIZE  : {cfg.img_size}×{cfg.img_size}')
print(f'  CLASSES   : {num_classes}')
print('-' * 65)
print(f'  Total params    : {total_params:>12,}  ({total_params/1e6:.2f} M)')
print(f'  Trainable now   : {trainable_params:>12,}  ({trainable_params/1e6:.2f} M)  [head only during warmup]')
print(f'  Frozen (warmup) : {frozen_params:>12,}  ({frozen_params/1e6:.2f} M)')
print('=' * 65)

# Count layers by type
layer_counts = {}
for name, module in model.named_modules():
    t = type(module).__name__
    layer_counts[t] = layer_counts.get(t, 0) + 1

print('\nLayer breakdown:')
for t, count in sorted(layer_counts.items(), key=lambda x: -x[1]):
    if count > 0 and t not in ('Sequential',):
        print(f'  {t:<30} × {count}')

if HAS_TORCHINFO:
    print('\nDetailed torchinfo summary:')
    torchinfo_summary(
        model,
        input_size=(1, 3, cfg.img_size, cfg.img_size),
        device=DEVICE,
        col_names=['input_size', 'output_size', 'num_params', 'trainable'],
        depth=3,
        verbose=1,
    )
else:
    print('\n[Install torchinfo for a detailed layer table: pip install torchinfo]')


In [ ]:
# ── Optimiser ─────────────────────────────────────────────────────────────
# Only head params are active initially (backbone is frozen)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

def lr_lambda(epoch):
    '''Linear warmup → cosine decay.'''
    if epoch < cfg.warmup_epochs:
        return float(epoch + 1) / float(max(1, cfg.warmup_epochs))
    progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))   # floor at 5% LR

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# SWA model + scheduler (activated after cfg.swa_start epoch)
swa_model     = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=cfg.swa_lr, anneal_epochs=5)
swa_active    = False

# AMP scaler
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

print('Optimiser  : AdamW (lr={}, wd={})'.format(cfg.lr, cfg.weight_decay))
print('Scheduler  : LambdaLR (linear warmup + cosine)')
print('SWA        : enabled={}, start_epoch={}, swa_lr={}'.format(
    cfg.use_swa, cfg.swa_start, cfg.swa_lr))
print('Grad accum : every {} steps (eff. batch = {})'.format(
    cfg.accum_steps, cfg.batch_size * cfg.accum_steps))


In [ ]:
# ── MixUp + CutMix ────────────────────────────────────────────────────────

def mixup_data(x, y, alpha=0.4):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def cutmix_data(x, y, alpha=1.0):
    '''CutMix: paste a rectangular patch from a shuffled image.'''
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape

    # Sample patch size
    cut_ratio = math.sqrt(1.0 - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cx = random.randint(0, W)
    cy = random.randint(0, H)
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, W)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, H)

    mixed = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam_actual = 1.0 - (y2 - y1) * (x2 - x1) / float(H * W)
    return mixed, y, y[idx], lam_actual


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('MixUp + CutMix helpers ready.')


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None,
              mixup_alpha=0.0, cutmix_alpha=0.0, cutmix_prob=0.5,
              accum_steps=1):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss, running_correct, total = 0.0, 0, 0
    y_true_all, y_pred_all = [], []

    pbar = tqdm(enumerate(loader), total=len(loader),
                desc='train' if is_train else 'eval ', leave=False)

    for step, (x, y) in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        # ── MixUp / CutMix ─────────────────────────────────────
        if is_train and (mixup_alpha > 0 or cutmix_alpha > 0):
            if random.random() < cutmix_prob and cutmix_alpha > 0:
                x, y_a, y_b, lam = cutmix_data(x, y, alpha=cutmix_alpha)
            else:
                x, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)
            use_mixed = True
        else:
            y_a, y_b, lam = y, y, 1.0
            use_mixed = False

        # ── Forward ─────────────────────────────────────────────
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x)
            loss = (mixed_criterion(criterion, logits, y_a, y_b, lam)
                    if use_mixed else criterion(logits, y_a))
            loss = loss / accum_steps   # scale for gradient accumulation

        # ── Backward ────────────────────────────────────────────
        if is_train:
            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        preds = logits.argmax(dim=1)
        running_loss    += loss.item() * accum_steps * x.size(0)
        running_correct += (preds == y_a).sum().item()
        total           += x.size(0)

        y_true_all.append(y_a.detach().cpu())
        y_pred_all.append(preds.detach().cpu())

        pbar.set_postfix(loss=f'{running_loss/total:.4f}',
                         acc=f'{running_correct/total:.4f}')

    y_true = torch.cat(y_true_all).numpy() if y_true_all else np.array([])
    y_pred = torch.cat(y_pred_all).numpy() if y_pred_all else np.array([])
    return running_loss / max(total, 1), running_correct / max(total, 1), y_true, y_pred


def per_class_accuracy(y_true, y_pred, n):
    return {c: float((y_pred[y_true == c] == c).mean())
            if (y_true == c).sum() > 0 else float('nan')
            for c in range(n)}


def plot_confusion_matrix(y_true, y_pred, names, save_path=None):
    if not HAS_SKLEARN:
        return
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — Test Set')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


print('Training helpers ready.')


In [ ]:
history = []
best_state = None
best_val_acc = -np.inf
patience_counter = 0
backbone_unfrozen = False

start = time.time()
for epoch in range(1, cfg.epochs + 1):

    # ── Unfreeze backbone after warmup ────────────────────────────────────
    if epoch == cfg.freeze_backbone_epochs + 1 and not backbone_unfrozen:
        for p in _backbone_module.parameters():
            p.requires_grad = True
        backbone_unfrozen = True

        # Differential LR: backbone gets 10× lower LR than head
        backbone_params = list(_backbone_module.parameters())
        backbone_ids    = {id(p) for p in backbone_params}
        head_params     = [p for p in model.parameters() if id(p) not in backbone_ids]

        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': cfg.lr * 0.1},
            {'params': head_params,     'lr': cfg.lr * 0.5},
        ], weight_decay=cfg.weight_decay)
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

        # Re-init SWA model with updated parameter set
        swa_model = AveragedModel(model)

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f'Backbone unfrozen | trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params')

    # ── SWA activation ───────────────────────────────────────────────────
    if cfg.use_swa and epoch == cfg.swa_start and backbone_unfrozen:
        swa_active = True
        print(f'SWA activated at epoch {epoch}.')

    # ── Train / Val ───────────────────────────────────────────────────────
    train_loss, train_acc, _, _ = run_epoch(
        model, train_loader, criterion, optimizer, scaler,
        mixup_alpha=cfg.mixup_alpha,
        cutmix_alpha=cfg.cutmix_alpha,
        cutmix_prob=cfg.cutmix_prob,
        accum_steps=cfg.accum_steps,
    )
    val_loss, val_acc, yv, pv = run_epoch(model, val_loader, criterion)

    # ── SWA update or normal LR step ─────────────────────────────────────
    if swa_active:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    row = dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
               val_loss=val_loss, val_acc=val_acc, lr=current_lr,
               swa=swa_active)
    history.append(row)

    print(
        f'Epoch {epoch:02d}/{cfg.epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | '
        f'lr={current_lr:.2e}' +
        (' [SWA]' if swa_active else '')
    )

    # ── Checkpoint ───────────────────────────────────────────────────────
    if (val_acc - best_val_acc) > cfg.min_delta:
        best_val_acc = val_acc
        best_state = {
            'model':        copy.deepcopy(model.state_dict()),
            'epoch':        epoch,
            'val_acc':      val_acc,
            'config':       asdict(cfg),
            'backbone':     BACKBONE_CHOICE,
            'class_names':  class_names,
            'class_to_idx': base_dataset.class_to_idx,
        }
        torch.save(best_state, RUN_DIR / 'best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= cfg.patience:
        print(f'Early stopping at epoch {epoch}.')
        break

# ── SWA finalisation ─────────────────────────────────────────────────────
if swa_active:
    print('\nUpdating SWA BatchNorm statistics...')
    update_bn(train_loader, swa_model, device=DEVICE)
    _, swa_val_acc, _, _ = run_epoch(swa_model, val_loader, criterion)
    print(f'SWA val acc: {swa_val_acc:.4f}  (best single-model: {best_val_acc:.4f})')

    if swa_val_acc > best_val_acc:
        print('SWA model is better — saving as best checkpoint.')
        torch.save({
            'model':        swa_model.module.state_dict(),
            'epoch':        epoch,
            'val_acc':      swa_val_acc,
            'config':       asdict(cfg),
            'backbone':     BACKBONE_CHOICE,
            'class_names':  class_names,
            'class_to_idx': base_dataset.class_to_idx,
            'swa':          True,
        }, RUN_DIR / 'best_model.pt')
        best_val_acc = swa_val_acc

elapsed = time.time() - start
print(f'\nTraining done in {elapsed/60:.1f} min | Best val acc: {best_val_acc:.4f}')

history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / 'train_history.csv', index=False)
display(history_df.tail(10))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train')
axes[0].plot(history_df['epoch'], history_df['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_acc'], label='Train')
axes[1].plot(history_df['epoch'], history_df['val_acc'],   label='Val')
axes[1].axvline(x=cfg.swa_start, color='red', linestyle='--', alpha=0.5, label='SWA start')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['lr'])
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

# Shade SWA region
for ax in axes:
    if history_df['swa'].any():
        swa_start_epoch = history_df[history_df['swa']]['epoch'].min()
        ax.axvspan(swa_start_epoch, history_df['epoch'].max(),
                   alpha=0.08, color='green', label='SWA active')

plt.tight_layout()
plt.savefig(RUN_DIR / 'training_curves.png', dpi=150)
plt.show()


In [ ]:
ckpt = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]} | val_acc={ckpt["val_acc"]:.4f}'
      + (' (SWA)' if ckpt.get('swa') else ''))

test_loss, test_acc, yt, pt = run_epoch(model, test_loader, criterion)
print(f'\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}')

# ── Overfitting summary ───────────────────────────────────────────────────
last = history_df.iloc[-1]
print(f'\n--- Overfitting check ---')
print(f'Train acc: {last.train_acc:.4f}  |  Val acc: {last.val_acc:.4f}'
      f'  |  Gap: {last.train_acc - last.val_acc:.4f}')
print(f'Train loss: {last.train_loss:.4f}  |  Val loss: {last.val_loss:.4f}'
      f'  |  Ratio: {last.val_loss / max(last.train_loss, 1e-8):.2f}x')
if last.train_acc - last.val_acc < 0.05:
    print('✅ Good generalisation (gap < 5%)')
elif last.train_acc - last.val_acc < 0.10:
    print('⚠️  Mild overfitting (gap 5-10%)')
else:
    print('❌ Significant overfitting (gap > 10%) — consider more dropout or data')

# ── Per-class accuracy ────────────────────────────────────────────────────
pc_acc = per_class_accuracy(yt, pt, num_classes)
pc_df = pd.DataFrame({
    'class':           class_names,
    'n_test':          np.bincount(test_targets, minlength=num_classes),
    'test_class_acc':  [round(pc_acc[i], 4) for i in range(num_classes)],
})
display(pc_df.sort_values('test_class_acc'))
pc_df.to_csv(RUN_DIR / 'test_per_class_accuracy.csv', index=False)

if HAS_SKLEARN:
    print('\nClassification Report:')
    print(classification_report(yt, pt, target_names=class_names, digits=3))
    pd.DataFrame(
        classification_report(yt, pt, target_names=class_names, digits=3, output_dict=True)
    ).T.to_csv(RUN_DIR / 'classification_report.csv')

plot_confusion_matrix(yt, pt, class_names, save_path=RUN_DIR / 'confusion_matrix.png')


In [ ]:
export = {
    'model_state_dict': model.state_dict(),
    'class_names':      class_names,
    'class_to_idx':     base_dataset.class_to_idx,
    'img_size':         cfg.img_size,
    'normalize_mean':   MEAN,
    'normalize_std':    STD,
    'architecture':     BACKBONE_CHOICE,
    'val_acc':          float(best_val_acc),
    'test_acc':         float(test_acc),
}
torch.save(export, RUN_DIR / 'model_inference.pt')

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

print('Saved artifacts:')
for p in sorted(RUN_DIR.glob('*')):
    print(f'  {p.name:<42} {p.stat().st_size/1024:>8.1f} KB')


In [ ]:
if IS_COLAB:
    from google.colab import files
    import zipfile

    bundle = f'/content/{cfg.run_name}_artifacts.zip'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(RUN_DIR.glob('*')):
            zf.write(p, arcname=p.name)
    print(f'Downloading {bundle} ...')
    files.download(bundle)
else:
    print('Local run — artifacts at:')
    print(f'  {RUN_DIR}')


In [ ]:
def predict_image(image_path: str, checkpoint_path: str = None,
                  use_tta: bool = True) -> dict:
    '''Predict skin condition for a single image with optional TTA.

    Args:
        image_path:      Path to image file.
        checkpoint_path: Optional explicit path to model_inference.pt.
        use_tta:         Average over 3 augmented views for more robust prediction.

    Returns:
        dict: pred_class, confidence, tta_views, top3
    '''
    checkpoint_path = checkpoint_path or str(RUN_DIR / 'model_inference.pt')
    blob = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

    # Rebuild model
    inf_model, _ = build_model(blob['architecture'], len(blob['class_names']), DEVICE)
    inf_model.load_state_dict(blob['model_state_dict'])
    inf_model.eval()

    im = Image.open(image_path).convert('RGB')
    views = tta_tfms if use_tta else [eval_tfms]

    all_probs = []
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
        for tfm in views:
            x = tfm(im).unsqueeze(0).to(DEVICE)
            probs = torch.softmax(inf_model(x), dim=1)[0]
            all_probs.append(probs)

    avg_probs = torch.stack(all_probs).mean(dim=0)
    conf, idx = torch.max(avg_probs, dim=0)
    names = blob['class_names']

    top3 = sorted(enumerate(avg_probs.cpu().tolist()),
                  key=lambda z: z[1], reverse=True)[:3]

    return {
        'pred_class': names[idx.item()],
        'confidence': round(float(conf.item()), 4),
        'tta_views':  len(views),
        'top3':       [(names[i], round(p, 4)) for i, p in top3],
    }


# Example (uncomment to run):
# result = predict_image('path/to/skin_image.jpg', use_tta=True)
# print(result)
